# Diagnóstico controlado: levodopa → dopamina → m-tiramina

Este notebook **no modifica los modelos originales** ni vuelve a introducir reacciones `EX_`. Prueba por separado (1) el par sin cambios, (2) un control positivo que añade dopamina a los seeds y (3) un **transporte de levodopa hipotético** en *E. faecalis* para examinar el bloqueo topológico.

**Importante:** el transporte hipotético `HYP_LEVDOPA_IMPORT` es una intervención de diagnóstico, **no una reacción respaldada por AGORA2 ni evidencia experimental**. Si funciona, prueba que el bloqueo es consistente con la falta de una vía de importación que permita iniciar el ciclo; no valida automáticamente la biología.

Puedes ejecutar desde `notebooks/` o desde la carpeta raíz del proyecto: el código busca automáticamente `sbml_sin_ex/` entre los directorios superiores, y desde ahí localiza `m2m/seeds.sbml` y `m2m_targets/seeds.sbml`. Usa un entorno que tenga `cobra`, `m2m` y `miscoto` instalados.

In [9]:
import sys
import shutil
import subprocess
import importlib.metadata as md

print("Python:", sys.version)
print("Clingo:", shutil.which("clingo"))
print("M2M:", shutil.which("m2m"))

for pkg in ["Metage2Metabo", "miscoto", "clyngor"]:
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "NO INSTALADO")

if shutil.which("clingo"):
    p = subprocess.run(
        ["clingo", "--version"],
        capture_output=True,
        text=True
    )
    print(p.stdout)

Python: 3.14.7 | packaged by conda-forge | (main, Aug 31 2026, 07:13:54) [Clang 20.1.8 ]
Clingo: /Users/javigajardo/miniforge3/envs/opti_redes/bin/clingo
M2M: /Users/javigajardo/miniforge3/envs/opti_redes/bin/m2m
Metage2Metabo 1.6.3
miscoto 3.2.0
clyngor 1.0.0
clingo version 5.8.0
Address model: 64-bit

libclingo version 5.8.0
Configuration: with Python 3.14.0, without Lua

libclasp version 3.4.0 (libpotassco version 1.2.0)
Configuration: WITH_THREADS=1
Copyright (C) Benjamin Kaufmann

License: The MIT License <https://opensource.org/licenses/MIT>



In [10]:
from pathlib import Path
import subprocess
import json
import shutil
import miscoto

# Instancia que generó M2M antes de fallar
folder = (
    Path.home()
    / "Desktop/proyecto_redes/diagnostico_levodopa"
    / "control_sin_dopamina/community_analysis"
)

archivos = list(folder.glob("miscoto_*.lp"))
assert archivos, "No se encontró la instancia ASP"

lp = max(archivos, key=lambda p: p.stat().st_mtime)

encoding = (
    Path(miscoto.__file__).resolve().parent
    / "encodings/community_soup.lp"
)

print("Instancia:", lp.name)
print("Tamaño:", lp.stat().st_size, "bytes")
print("Encoding existe:", encoding.exists())

clingo = shutil.which("clingo")
assert clingo, "No se encontró Clingo"

p = subprocess.run(
    [
        clingo,
        str(lp),
        str(encoding),
        "--outf=2",
        "-n", "1"
    ],
    capture_output=True,
    text=True,
    timeout=180
)

print("Código de salida:", p.returncode)
print("Errores:", p.stderr[-2000:])

try:
    data = json.loads(p.stdout)
    print("RESULTADO:", data.get("Result"))
    print("MODELOS:", data.get("Models"))
    for call in data.get("Call", []):
        for witness in call.get("Witnesses", [])[:1]:
            print("COSTOS:", witness.get("Costs"))
except json.JSONDecodeError:
    print("Salida:", p.stdout[-2000:])

Instancia: miscoto_yg393tv7.lp
Tamaño: 1936 bytes
Encoding existe: True
Código de salida: 0
Errores: 

/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp:36:16-24: info: atom does not occur in any rule head:
  draft(O)

/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp:37:16-27: info: atom does not occur in any rule head:
  bacteria(O)

/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp:43:30-44: info: atom does not occur in any rule head:
  product(M,R,O)

/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp:43:46-59: info: atom does not occur in any rule head:
  reaction(R,O)

/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp:44:28-44: info: atom does not occur in any rule head:
  reactant

In [11]:
from pathlib import Path
from miscoto import sbml as miscoto_sbml
import traceback
import xml.etree.ElementTree as ET

# 1. Revisar qué contiene la instancia ASP
lp = max(
    (WORK / "control_sin_dopamina" / "community_analysis")
    .glob("miscoto_*.lp"),
    key=lambda p: p.stat().st_mtime
)

lines = lp.read_text().splitlines()

print("=== CONTENIDO DE LA INSTANCIA ===")

for pred in ["seed", "target", "bacteria",
             "reaction", "reactant", "product"]:
    n = sum(
        line.strip().startswith(pred + "(")
        for line in lines
    )
    print(f"{pred}: {n}")

# 2. Leer cada modelo directamente con el parser de MiSCoTo
print("\n=== PARSER DE MiSCoTo ===")

for path in sorted(pair_clean.glob("*.xml")):
    print(f"\nMODELO: {path.name}")

    try:
        model = miscoto_sbml.readSBMLnetwork_symbionts_clyngor(
            str(path),
            path.stem
        )
        print("PARSER OK")
        print("Número de hechos ASP:", len(model))

    except BaseException as e:
        print("ERROR DETECTADO")
        print("Tipo:", type(e).__name__)
        print("Mensaje:", str(e))
        traceback.print_exception(
            type(e), e, e.__traceback__, limit=5
        )

# 3. Verificar atributos SBML
print("\n=== METADATOS SBML ===")

for path in sorted(pair_clean.glob("*.xml")):
    root = ET.parse(path).getroot()

    species = [
        e for e in root.iter()
        if e.tag.rsplit("}", 1)[-1] == "species"
    ]

    reactions = [
        e for e in root.iter()
        if e.tag.rsplit("}", 1)[-1] == "reaction"
    ]

    missing_names = [
        e.get("id")
        for e in species
        if e.get("name") is None
    ]

    print(f"\n{path.name}")
    print("Metabolitos:", len(species))
    print("Reacciones:", len(reactions))
    print("Metabolitos sin name:", len(missing_names))
    print("Ejemplos:", missing_names[:5])

=== CONTENIDO DE LA INSTANCIA ===
seed: 104
target: 1
bacteria: 0
reaction: 0
reactant: 0
product: 0

=== PARSER DE MiSCoTo ===

MODELO: Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml
ERROR DETECTADO
Tipo: SystemExit
Mensaje: Empty name field for species in the following SBML: /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/par_limpio/Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml

MODELO: Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml
ERROR DETECTADO
Tipo: SystemExit
Mensaje: Empty name field for species in the following SBML: /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/par_limpio/Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml

=== METADATOS SBML ===

Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml
Metabolitos: 968
Reacciones: 900
Metabolitos sin name: 4
Ejemplos: ['M_pe180_c', 'M_peai17_c', 'M_pei17_c', 'M_pg180_c']

Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml
Metabolitos: 1486
Reacciones: 1375
Metabolitos sin name: 4
Ejemplos: ['M_pe180_c', 'M_pea

Traceback (most recent call last):
  File "/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/sbml.py", line 255, in readSBMLnetwork_symbionts_clyngor
    speciesNm = e.attrib.get("name").replace('"','')
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'replace'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/var/folders/sm/fx2cw8ln4q36c4l8gmpzt4br0000gn/T/ipykernel_45933/3541548272.py", line 32, in <module>
    model = miscoto_sbml.readSBMLnetwork_symbionts_clyngor(
        str(path),
        path.stem
    )
  File "/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/sbml.py", line 257, in readSBMLnetwork_symbionts_clyngor
    sys.exit("Empty name field for species in the following SBML: " + filename)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
SystemExit: Empty name field for

In [13]:
from pathlib import Path
from lxml import etree
from miscoto import sbml as miscoto_sbml

def corregir_nombres(carpeta_entrada, carpeta_salida):
    carpeta_entrada = Path(carpeta_entrada)
    carpeta_salida = Path(carpeta_salida)
    carpeta_salida.mkdir(parents=True, exist_ok=True)

    archivos = sorted(carpeta_entrada.glob("*.xml"))

    for archivo in archivos:
        tree = etree.parse(str(archivo))

        especies = tree.xpath(
            "//*[local-name()='listOfSpecies']/*[local-name()='species']"
        )

        corregidos = 0

        for especie in especies:
            nombre = especie.get("name")

            if nombre is None or not nombre.strip():
                identificador = especie.get("id")

                if not identificador:
                    raise ValueError(
                        f"Metabolito sin ID en {archivo.name}"
                    )

                especie.set("name", identificador)
                corregidos += 1

        destino = carpeta_salida / archivo.name

        tree.write(
            str(destino),
            encoding="UTF-8",
            xml_declaration=True
        )

        print(f"\n{archivo.name}")
        print(f"Metabolitos sin nombre corregidos: {corregidos}")

        # Verificar que MiSCoTo pueda leer el SBML
        miscoto_sbml.readSBMLnetwork_symbionts_clyngor(
            str(destino),
            destino.stem
        )

        print("Parser MiSCoTo: OK")

    return carpeta_salida


# Corregir el par bacteriano
pair_fixed = corregir_nombres(
    pair_clean,
    WORK / "par_nombres_corregidos"
)


Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml
Metabolitos sin nombre corregidos: 4
Parser MiSCoTo: OK

Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml
Metabolitos sin nombre corregidos: 4
Parser MiSCoTo: OK


In [14]:
from cobra.io import read_sbml_model

def firma_reacciones(model):
    return {
        r.id: (
            r.lower_bound,
            r.upper_bound,
            tuple(sorted(
                (m.id, coef)
                for m, coef in r.metabolites.items()
            ))
        )
        for r in model.reactions
    }

for original in sorted(pair_clean.glob("*.xml")):
    corregido = pair_fixed / original.name

    modelo_original = read_sbml_model(str(original))
    modelo_corregido = read_sbml_model(str(corregido))

    assert firma_reacciones(modelo_original) == firma_reacciones(modelo_corregido)
    assert (
        {m.id for m in modelo_original.metabolites}
        ==
        {m.id for m in modelo_corregido.metabolites}
    )

    print(original.name, "OK: reacciones y metabolitos conservados")

Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml OK: reacciones y metabolitos conservados
Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml OK: reacciones y metabolitos conservados


In [15]:
result_fixed = mincom(
    pair_fixed,
    SEEDS,
    "control_nombres_corregidos"
)


$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/par_nombres_corregidos -s /Users/javigajardo/Desktop/proyecto_redes/m2m/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_nombres_corregidos
######### Creating metabolic instance for the whole community #########
Created temporary instance file in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_nombres_corregidos/community_analysis/miscoto_3wm1zs8_.lp

###############################################
#                                             #
#         Minimal community selection         #
#                                             #
###############################################

Running minimal community selection
/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp

In the initial and minimal communities 0 targets a

In [20]:
import cobra

# Crear el par bacteriano para el experimento
pair_patch = create_pair("par_con_importacion_hipotetica")

# Modificar solamente E. faecalis
patch_path = pair_patch / EFA
m = cobra.io.read_sbml_model(str(patch_path))

lev_e = m.metabolites.get_by_id("levdopa_e")
lev_c = m.metabolites.get_by_id("levdopa_c")

# Añadir transporte hipotético
r = cobra.Reaction("HYP_LEVDOPA_IMPORT")
r.name = "Transporte hipotetico de levodopa"
r.lower_bound = 0
r.upper_bound = 1000

r.add_metabolites({
    lev_e: -1,
    lev_c: 1
})

m.add_reactions([r])
cobra.io.write_sbml_model(m, str(patch_path))

# Corregir los nombres SBML antes de ejecutar MiSCoTo
pair_patch_fixed = corregir_nombres(
    pair_patch,
    WORK / "par_importacion_nombres_corregidos"
)

# Ejecutar mincom con el seed ORIGINAL
result_patch = mincom(
    pair_patch_fixed,
    SEEDS,
    "control_importacion_hipotetica"
)

print("\nRESULTADO FINAL")
print("Producible:", result_patch.get("producible"))
print("Comunidad:", result_patch.get("bacteria"))
print("Esenciales:", result_patch.get("essential_symbionts"))


Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml
Metabolitos sin nombre corregidos: 4
Parser MiSCoTo: OK

Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml
Metabolitos sin nombre corregidos: 4
Parser MiSCoTo: OK

$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/par_importacion_nombres_corregidos -s /Users/javigajardo/Desktop/proyecto_redes/m2m/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_importacion_hipotetica
######### Creating metabolic instance for the whole community #########
Created temporary instance file in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_importacion_hipotetica/community_analysis/miscoto_346ewsyz.lp

###############################################
#                                             #
#         Minimal community selection         #
#                                             #
###################

In [18]:
# Control 1: Dopamina extracelular
seed_dopamine_e = create_seed(
    base_seed_ids | {"M_dpmn_e"},
    "seed_dopamina_extracelular"
)

result_dopamine_e = mincom(
    pair_fixed,
    seed_dopamine_e,
    "control_positivo_dopamina_extracelular"
)

# Control 2: Dopamina citosólica
seed_dopamine_c = create_seed(
    base_seed_ids | {"M_dpmn_c"},
    "seed_dopamina_citosolica"
)

result_dopamine_c = mincom(
    pair_fixed,
    seed_dopamine_c,
    "control_positivo_dopamina_citosolica"
)


$ m2m seeds --metabolites /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/seed_dopamina_extracelular/metabolites.txt -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/seed_dopamina_extracelular
Seeds SBML file created in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/seed_dopamina_extracelular/seeds.sbml
--- Total runtime 0.00 seconds ---
--- Logs written in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/seed_dopamina_extracelular/m2m_seeds.log ---



$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/par_nombres_corregidos -s /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/seed_dopamina_extracelular/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_positivo_dopamina_extracelular
######### Creating metabolic instance for the whole community #########
Created temporary instance fi

In [19]:
for nombre, resultado in [
    ("Sin dopamina", result_fixed),
    ("Dopamina extracelular", result_dopamine_e),
    ("Dopamina citosólica", result_dopamine_c),
]:
    print(f"\n=== {nombre} ===")
    print("Producible:", resultado.get("producible"))
    print("No producible:", resultado.get("still_unprod"))
    print("Comunidad:", resultado.get("bacteria"))


=== Sin dopamina ===
Producible: []
No producible: ['M_mtym_e']
Comunidad: []

=== Dopamina extracelular ===
Producible: ['M_mtym_e']
No producible: []
Comunidad: ['Eggerthella_lenta_DSM_2243_AGORA2_v2.01']

=== Dopamina citosólica ===
Producible: ['M_mtym_e']
No producible: []
Comunidad: ['Eggerthella_lenta_DSM_2243_AGORA2_v2.01']


In [5]:
from pathlib import Path
import subprocess
import sys
import shutil

print("Python actual:", sys.executable)
print("M2M disponible:", shutil.which("m2m"))

M2M_BIN = Path("/opt/miniconda3/envs/opti_redes/bin/m2m")

print("M2M original existe:", M2M_BIN.exists())

if M2M_BIN.exists():
    resultado = subprocess.run(
        [str(M2M_BIN), "--help"],
        capture_output=True,
        text=True
    )
    print(resultado.stdout[:1000])
    print(resultado.stderr[:1000])

Python actual: /Users/javigajardo/miniforge3/envs/opti_redes/bin/python
M2M disponible: /Users/javigajardo/miniforge3/envs/opti_redes/bin/m2m
M2M original existe: False


In [6]:
import json
import shutil
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path
import cobra

# Localiza la raiz del proyecto aunque ejecutes este notebook desde notebooks/
CANDIDATES = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
BASE = next(
    (p for p in CANDIDATES if (p / "sbml_sin_ex").is_dir()),
    None
)
if BASE is None:
    raise FileNotFoundError(
        "No encontre la carpeta sbml_sin_ex en el directorio actual ni en sus padres. "
        f"Directorio actual: {Path.cwd()}"
    )
print("Raiz del proyecto:", BASE)
SRC = BASE / "sbml_sin_ex"
SEEDS = BASE / "m2m" / "seeds.sbml"
TARGETS = BASE / "m2m_targets" / "seeds.sbml"
WORK = BASE / "diagnostico_levodopa"
WORK.mkdir(exist_ok=True)

EFA = "Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml"
ELENTA = "Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml"

for p in (SRC / EFA, SRC / ELENTA, SEEDS, TARGETS):
    if not p.exists():
        raise FileNotFoundError(f"Falta {p}; actualiza las rutas de configuración")

def sbml_species(sbml_path):
    root = ET.parse(sbml_path).getroot()
    return {node.get("id") for node in root.iter() if node.tag.rsplit("}", 1)[-1] == "species"}

base_seed_ids = sbml_species(SEEDS)
print("Número de semillas:", len(base_seed_ids))
for x in ("M_levdopa_e", "M_dpmn_e", "M_dpmn_c", "M_h_c", "M_h_e", "M_nadh_c", "M_mtym_e"):
    print(x, x in base_seed_ids)

def run(command):
    print("\n$", " ".join(map(str, command)))
    p = subprocess.run(list(map(str, command)), text=True, capture_output=True)
    print((p.stdout + "\n" + p.stderr)[-3500:])
    if p.returncode:
        raise RuntimeError(f"Falló comando con código {p.returncode}")
    return p

def create_seed(ids, label):
    folder = WORK / label
    folder.mkdir(exist_ok=True)
    txt = folder / "metabolites.txt"
    txt.write_text("\n".join(sorted(ids)) + "\n")
    run(["m2m", "seeds", "--metabolites", txt, "-o", folder])
    return folder / "seeds.sbml"

def create_pair(label):
    folder = WORK / label
    folder.mkdir(exist_ok=True)
    for name in (EFA, ELENTA):
        shutil.copy2(SRC / name, folder / name)
    return folder

def mincom(folder, seed, label):
    output = WORK / label
    if output.exists():
        shutil.rmtree(output)
    run(["m2m", "mincom", "-n", folder, "-s", seed, "-t", TARGETS, "-o", output])
    path = output / "community_analysis" / "mincom.json"
    data = json.loads(path.read_text())
    print("\nRESULTADO:", {k:data.get(k) for k in ("producible","still_unprod","bacteria","essential_symbionts")})
    return data

Raiz del proyecto: /Users/javigajardo/Desktop/proyecto_redes
Número de semillas: 104
M_levdopa_e True
M_dpmn_e False
M_dpmn_c False
M_h_c True
M_h_e True
M_nadh_c True
M_mtym_e False


## 1. Auditar las reacciones después de la limpieza

Comprobamos las reacciones necesarias; en particular `EX_dopa_e_` no debe estar presente.

In [7]:
for filename in (EFA, ELENTA):
    m = cobra.io.read_sbml_model(str(SRC / filename))
    print("\n===", filename, "===")
    for rxn in m.reactions:
        if any(x in rxn.id for x in ("3HLYTCL", "DOPA34DHPHEt", "DOPAENT4tc", "DOPADH", "MTYMt2r", "EX_dopa")):
            print(rxn.id, ":", rxn.reaction, "bounds:", rxn.bounds)


=== Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml ===
3HLYTCL : h_c + levdopa_c --> co2_c + dpmn_c bounds: (0.0, 1000.0)
DOPA34DHPHEt : dpmn_c + levdopa_e <=> dpmn_e + levdopa_c bounds: (-1000.0, 1000.0)
DOPA34DHPHEt6 : dpmn_c + h_c + levdopa_e + na1_e <=> dpmn_e + h_e + levdopa_c + na1_c bounds: (-1000.0, 1000.0)

=== Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml ===
DOPAENT4tc : dpmn_e + h_e <=> dpmn_c + h_c bounds: (-1000.0, 1000.0)
DOPADH : dpmn_c + h_c + nadh_c --> h2o_c + mtym_c + nad_c bounds: (0.0, 1000.0)
MTYMt2r : h_e + mtym_e <=> h_c + mtym_c bounds: (-1000.0, 1000.0)


## 2. Control negativo: par original limpio, sin dopamina en seeds

Se espera que `M_mtym_e` **no** sea producible, si el ciclo de importación está bloqueado.

In [8]:
pair_clean = create_pair("par_limpio")
result_negative = mincom(pair_clean, SEEDS, "control_sin_dopamina")


$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/par_limpio -s /Users/javigajardo/Desktop/proyecto_redes/m2m/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_sin_dopamina
######### Creating metabolic instance for the whole community #########
Created temporary instance file in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa/control_sin_dopamina/community_analysis/miscoto_yg393tv7.lp

###############################################
#                                             #
#         Minimal community selection         #
#                                             #
###############################################

Running minimal community selection
/Users/javigajardo/miniforge3/envs/opti_redes/lib/python3.14/site-packages/miscoto/encodings/community_soup.lp

Traceback (most recent call last):
  File "/Users/javigajardo/miniforge3/e

RuntimeError: Falló comando con código 1

## 3. Control positivo: agregar dopamina extracelular **solo como diagnóstico**

Si el transporte `DOPAENT4tc` y la reacción `DOPADH` funcionan en la lectura de MiSCoTo, la cepa *E. lenta* DSM 2243 debería poder producir `M_mtym_e` cuando existe `M_dpmn_e` junto a los cofactores ya presentes. **No usar este seed para concluir cooperación mínima**, pues elimina la necesidad de un productor de dopamina.

In [ ]:
seed_dopamine_e = create_seed(base_seed_ids | {"M_dpmn_e"}, "seed_dopamina_extracelular")
result_dopamine_e = mincom(pair_clean, seed_dopamine_e, "control_positivo_dopamina_extracelular")

## 4. Segundo control positivo: dopamina citosólica

Útil únicamente si el paso anterior falló. Permite distinguir si la pérdida ocurre en el transportador de dopamina o en `DOPADH` / el exportador de m-tiramina.

In [ ]:
seed_dopamine_c = create_seed(base_seed_ids | {"M_dpmn_c"}, "seed_dopamina_citosolica")
result_dopamine_c = mincom(pair_clean, seed_dopamine_c, "control_positivo_dopamina_citosolica")

## 5. Intervención de diagnóstico: añadir **un transportador hipotético** de levodopa solo en E. faecalis

Se modela `levdopa_e → levdopa_c`, sin creación de dopamina ni m-tiramina de la nada. La descarboxilación `3HLYTCL` puede entonces producir `dpmn_c`, que puede exportarse mediante el antiporter existente. *E. lenta* puede importar esa dopamina y convertirla a m-tiramina. Esto **no** debe interpretarse como una reconstrucción validada: necesitarías evidencia de que ese transporte opera en la cepa o presentarlo explícitamente como escenario de gap-filling.

In [ ]:
pair_patch = create_pair("par_con_importacion_hipotetica")
patch_path = pair_patch / EFA
m = cobra.io.read_sbml_model(str(patch_path))
assert "HYP_LEVDOPA_IMPORT" not in m.reactions
lev_e = m.metabolites.get_by_id("levdopa_e")
lev_c = m.metabolites.get_by_id("levdopa_c")
r = cobra.Reaction("HYP_LEVDOPA_IMPORT")
r.name = "Transporte hipotetico levodopa e -> c (SOLO DIAGNOSTICO)"
r.lower_bound = 0
r.upper_bound = 1000
r.add_metabolites({lev_e: -1, lev_c: 1})
m.add_reactions([r])
cobra.io.write_sbml_model(m, str(patch_path))
print("Reacción incorporada:", m.reactions.get_by_id("HYP_LEVDOPA_IMPORT").reaction)
result_patch = mincom(pair_patch, SEEDS, "control_importacion_hipotetica")

## 6. Interpretación

- **Control positivo con `M_dpmn_e` funciona**, pero el par original no: el bloqueo principal está antes de la disponibilidad de dopamina externa (importación/producción/exportación en la bacteria productora).
- **`M_dpmn_e` falla pero `M_dpmn_c` funciona:** revisar `DOPAENT4tc`, IDs de compartimentos y reversibilidad interpretada por MiSCoTo.
- **Ambos positivos fallan:** revisar el parsing de `DOPADH`/`MTYMt2r`, cofactores y la instancia `.lp`; no asumir que basta con incorporar un nuevo transportador.
- **Par con transporte hipotético produce `M_mtym_e` y selecciona dos bacterias:** prueba de concepto de que el bloqueo topológico explica el resultado, **no prueba experimental** del transportador.
- **Si `mtym_e` sigue sin producirse aun con el patch:** hay otra dependencia faltante (p. ej. cofactores, orientación de una reacción, IDs o representación de intercambio).

Los resultados de cada prueba quedan guardados en `diagnostico_levodopa/<nombre>/community_analysis/mincom.json`.